# Objective of This Notebook

The objective of this notebook is to clean, validate, and prepare the structured financial dataset generated in Notebook 1 for downstream KPI engineering and clustering analysis.

This notebook focuses on identifying and handling data quality issues commonly present in financial statement datasets, including missing values, sparse features, inconsistent metrics, and extreme outliers. The preprocessing steps implemented here aim to improve the reliability, interpretability, and stability of the subsequent machine learning workflow.

The final output of this notebook will be a cleaned and modelling-ready financial dataset suitable for KPI calculation, dimensionality reduction, and clustering experiments.

# **1. Notebook Setup**

### **1.1 Imports**

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

### **1.2 Notebook Configuration**

In [2]:
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

# Plotting style
plt.style.use("default")

# Figure sizing
plt.rcParams["figure.figsize"] = (12, 6)

### **1.3 Paths**

In [3]:
PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

SRC_DIR = PROJECT_ROOT / "src"

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

# **2. Dataset Overview**

### **2.1 Load Flattened Financial Dataset**

In [4]:
financials_df = pd.read_excel(
    INTERIM_DATA_DIR / "financials_with_metadata.xlsx"
)

print(f"Dataset shape: {financials_df.shape}")

financials_df.head()

Dataset shape: (4576, 360)


,Symbol,Company Name,Exchange,Income_Tax Effect Of Unusual Items,Income_Tax Rate For Calcs,Income_Normalized EBITDA,Income_Total Unusual Items,Income_Total Unusual Items Excluding Goodwill,Income_Net Income From Continuing Operation Net Minority Interest,Income_Reconciled Depreciation,Income_Reconciled Cost Of Revenue,Income_EBITDA,Income_EBIT,Income_Net Interest Income,Income_Interest Expense,Income_Interest Income,Income_Normalized Income,Income_Net Income From Continuing And Discontinued Operation,Income_Total Expenses,Income_Total Operating Income As Reported,Income_Diluted Average Shares,Income_Basic Average Shares,Income_Diluted EPS,Income_Basic EPS,Income_Diluted NI Availto Com Stockholders,Income_Net Income Common Stockholders,Income_Net Income,Income_Net Income Including Noncontrolling Interests,Income_Net Income Discontinuous Operations,Income_Net Income Continuous Operations,Income_Tax Provision,Income_Pretax Income,Income_Other Income Expense,Income_Other Non Operating Income Expenses,Income_Special Income Charges,Income_Other Special Charges,Income_Impairment Of Capital Assets,Income_Gain On Sale Of Security,Income_Net Non Operating Interest Income Expense,Income_Total Other Finance Cost,Income_Interest Expense Non Operating,Income_Interest Income Non Operating,Income_Operating Income,Income_Operating Expense,Income_Depreciation Amortization Depletion Income Statement,Income_Depreciation And Amortization In Income Statement,Income_Research And Development,Income_Selling General And Administration,Income_Selling And Marketing Expense,Income_General And Administrative Expense,Income_Other Gand A,Income_Gross Profit,Income_Cost Of Revenue,Income_Total Revenue,Income_Operating Revenue,Balance_Treasury Shares Number,Balance_Ordinary Shares Number,Balance_Share Issued,Balance_Net Debt,Balance_Total Debt,Balance_Tangible Book Value,Balance_Invested Capital,Balance_Working Capital,Balance_Net Tangible Assets,Balance_Capital Lease Obligations,Balance_Common Stock Equity,Balance_Total Capitalization,Balance_Total Equity Gross Minority Interest,Balance_Stockholders Equity,Balance_Gains Losses Not Affecting Retained Earnings,Balance_Other Equity Adjustments,Balance_Retained Earnings,Balance_Additional Paid In Capital,Balance_Capital Stock,Balance_Common Stock,Balance_Preferred Stock,Balance_Total Liabilities Net Minority Interest,Balance_Total Non Current Liabilities Net Minority Interest,Balance_Other Non Current Liabilities,Balance_Non Current Deferred Liabilities,Balance_Non Current Deferred Taxes Liabilities,Balance_Long Term Debt And Capital Lease Obligation,Balance_Long Term Capital Lease Obligation,Balance_Long Term Debt,Balance_Current Liabilities,Balance_Other Current Liabilities,Balance_Current Deferred Liabilities,Balance_Current Deferred Revenue,Balance_Current Debt And Capital Lease Obligation,Balance_Current Debt,Balance_Other Current Borrowings,Balance_Current Notes Payable,Balance_Payables And Accrued Expenses,Balance_Current Accrued Expenses,Balance_Payables,Balance_Total Tax Payable,Balance_Income Tax Payable,Balance_Accounts Payable,Balance_Total Assets,Balance_Total Non Current Assets,Balance_Other Non Current Assets,Balance_Non Current Deferred Assets,Balance_Non Current Deferred Taxes Assets,Balance_Goodwill And Other Intangible Assets,Balance_Other Intangible Assets,Balance_Goodwill,Balance_Net PPE,Balance_Accumulated Depreciation,Balance_Gross PPE,Balance_Leases,Balance_Other Properties,Balance_Machinery Furniture Equipment,Balance_Properties,Balance_Current Assets,Balance_Other Current Assets,Balance_Assets Held For Sale Current,Balance_Receivables,Balance_Taxes Receivable,Balance_Accounts Receivable,Balance_Allowance For Doubtful Accounts Receivable,Balance_Gross Accounts Receivable,Balance_Cash Cash Equivalents And Short Term Investments,Balance_Cash And Cash Equivalents,CashFlow_Free Cash Flow,CashFlow_Repurchase Of Capital Stock,CashFlow_Repayment Of Debt,CashFlow_Issuance Of Debt,CashFlow_Capital

### **2.2 Dataset description**

In [5]:
financials_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4576 entries, 0 to 4575
Columns: 360 entries, Symbol to Website
dtypes: float64(351), str(9)
memory usage: 12.6 MB


In [6]:
financials_df.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
Income_Tax Effect Of Unusual Items,4573.0,-3.105859e+08,1.463905e+10,-7.838690e+11,-1.765680e+06,0.000000e+00,0.000000e+00,2.348990e+11
Income_Tax Rate For Calcs,4569.0,1.716604e-01,1.038284e-01,0.000000e+00,9.300000e-02,2.100000e-01,2.330000e-01,4.000000e-01
Income_Normalized EBITDA,4048.0,3.270788e+10,1.788561e+12,-6.651988e+13,-5.521470e+06,6.408400e+07,6.144380e+08,7.343200e+13
Income_Total Unusual Items,3626.0,-2.167490e+09,8.837795e+10,-3.919345e+12,-2.876875e+07,-1.314000e+06,1.049750e+06,8.945870e+11
Income_Total Unusual Items Excluding Goodwill,3627.0,-2.167066e+09,8.836577e+10,-3.919345e+12,-2.878950e+07,-1.316000e+06,1.049500e+06,8.945870e+11
Income_Net Income From Continuing Operation Net Minority Interest,4572.0,-5.395741e+09,1.513552e+12,-9.938466e+13,-2.243575e+07,1.190650e+07,2.339192e+08,1.748600e+13
Income_Reconciled Depreciation,4472.0,2.465760e+10,6.815284e+11,-1.829000e+09,3.149193e+06,2.731600e+07,1.759378e+08,3.765300e+13
Income_Reconciled Cost Of Revenue,3860.0,1.265812e+11,2.982689e+12,-1.370920e+08,3.500772e+07,3.285620e+08,2.000906e+09,1.310891e+14
Income_EBITDA,4045.0,3.169482e+10,1.825854e+12,-7.043922e+13,-9.077760e+06,4.704764e+07,5.660210e+08,7.337000e+13
Income_EBIT,4136.0,5.696077e+09,1.486022e+12,-8.189647e+13,-1.823878e+07,1.691550e+07,3.813865e+08,3.571700e+13


# **3. Data Preprocessing**

### **3.1 Missing Value Analysis**

Here we group the columns depending on type to process easily

In [14]:
identifier_columns = [
    "Symbol",
    "Company Name",
    "Exchange",
]

metadata_columns = [
    "Sector",
    "Industry",
    "Country",
    "Employees",
    "MarketCap",
    "EnterpriseValue",
    "Currency",
    "QuoteType",
    "Website",
]

financial_columns = [
    col for col in financials_df.columns
    if col.startswith("Income_")
    or col.startswith("Balance_")
    or col.startswith("CashFlow_")
]

print(f"Identifier columns: {len(identifier_columns)}")
print(f"Metadata columns: {len(metadata_columns)}")
print(f"Financial columns: {len(financial_columns)}")

Identifier columns: 3
Metadata columns: 9
Financial columns: 348


In [15]:
missing_summary = (
    financials_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["Column", "Missing %"]

missing_summary.head(30)

,Column,Missing %
0,CashFlow_Dividends Paid Direct,100.000000
1,Balance_Restricted Common Stock,99.978147
2,CashFlow_Receiptsfrom Government Grants,99.956294
3,CashFlow_Dividends Received Direct,99.934441
4,Income_Depletion Income Statement,99.912587
5,CashFlow_Paymentson Behalfof Employees,99.890734
6,CashFlow_Change In Dividend Payable,99.868881
7,Balance_General Partnership Capital,99.847028
8,Income_Net Income From Tax Loss Carryforward,99.825175
9,CashFlow_Dividend Paid Cfo,99.825175


A lof of missing values for more obscure financial metrics; this is expected

In [16]:
financial_coverage = (
    financials_df[financial_columns]
    .notna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

financial_coverage.columns = ["Financial Feature", "Non-Missing Count"]

financial_coverage["Coverage %"] = (
    financial_coverage["Non-Missing Count"] / len(financials_df) * 100
)

financial_coverage.head(30)

,Financial Feature,Non-Missing Count,Coverage %
0,Income_Total Revenue,4576,100.000000
1,Income_Tax Effect Of Unusual Items,4573,99.934441
2,Income_Net Income From Continuing Operation Ne...,4572,99.912587
3,Income_Diluted NI Availto Com Stockholders,4572,99.912587
4,Income_Net Income Common Stockholders,4572,99.912587
5,Balance_Total Assets,4571,99.890734
6,Income_Net Income Including Noncontrolling Int...,4571,99.890734
7,Income_Net Income From Continuing And Disconti...,4571,99.890734
8,Balance_Common Stock Equity,4570,99.868881
9,Balance_Total Liabilities Net Minority Interest,4569,99.847028


### **3.2 Remove Sparse Financial Features**

In [17]:
MAX_MISSING_PCT = 80 # We can tweak this threshold as needed

financial_missing_pct = financials_df[financial_columns].isna().mean() * 100

financial_columns_to_keep = financial_missing_pct[
    financial_missing_pct <= MAX_MISSING_PCT
].index.tolist()

financial_columns_to_drop = financial_missing_pct[
    financial_missing_pct > MAX_MISSING_PCT
].index.tolist()

print(f"Financial columns before filtering: {len(financial_columns):,}")
print(f"Financial columns kept: {len(financial_columns_to_keep):,}")
print(f"Financial columns dropped: {len(financial_columns_to_drop):,}")

Financial columns before filtering: 348
Financial columns kept: 214
Financial columns dropped: 134


### **3.3 Sector and Coverage**

In [18]:
sector_counts = financials_df["Sector"].value_counts(dropna=False)
sector_counts

Sector
Technology                751
Healthcare                740
Financial Services        707
Industrials               660
Consumer Cyclical         527
Communication Services    253
Consumer Defensive        230
Basic Materials           219
Energy                    206
Real Estate               175
Utilities                  98
NaN                        10
Name: count, dtype: int64

In [19]:
industry_counts = financials_df["Industry"].value_counts(dropna=False)
industry_counts.head(30)

Industry
Biotechnology                               302
Banks - Regional                            296
Software - Application                      215
Software - Infrastructure                   163
Medical Devices                             129
Asset Management                            109
Capital Markets                              87
Aerospace & Defense                          77
Drug Manufacturers - Specialty & Generic     76
Specialty Industrial Machinery               73
Internet Content & Information               69
Information Technology Services              68
Semiconductors                               66
Packaged Foods                               63
Oil & Gas E&P                                62
Auto Parts                                   56
Specialty Chemicals                          55
Telecom Services                             54
Engineering & Construction                   53
Restaurants                                  52
Medical Instruments & Supplies 

### **3.4 Remove Constant Columns**

In [20]:
candidate_columns = metadata_columns + financial_columns_to_keep

available_candidate_columns = [
    col for col in candidate_columns
    if col in financials_df.columns
]

nunique_summary = (
    financials_df[available_candidate_columns]
    .nunique(dropna=False)
    .sort_values()
    .reset_index()
)

nunique_summary.columns = ["Column", "Unique Values"]

nunique_summary.head(30)

,Column,Unique Values
0,Currency,1
1,QuoteType,1
2,Sector,12
3,Country,58
4,Industry,146
5,Balance_Properties,222
6,Balance_Preferred Stock,383
7,CashFlow_Sale Of Business,678
8,CashFlow_Purchase Of Intangibles,763
9,CashFlow_Net Intangibles Purchase And Sale,816


All values are in USD, so no currency conversion is needed

In [21]:
constant_columns = nunique_summary[
    nunique_summary["Unique Values"] <= 1
]["Column"].tolist()

print(f"Constant columns to drop: {len(constant_columns)}")
constant_columns

Constant columns to drop: 2


['Currency', 'QuoteType']

Luckily, only 2 columns were dropped here

In [22]:
metadata_columns_to_keep = [
    "Sector",
    "Industry",
    "Employees",
    "MarketCap",
    "EnterpriseValue",
    "Currency",
]

base_columns_to_keep = [
    col for col in identifier_columns + metadata_columns_to_keep
    if col in financials_df.columns
]

columns_to_keep = base_columns_to_keep + financial_columns_to_keep

clean_financials_df = financials_df[columns_to_keep].copy()

clean_financials_df = clean_financials_df.drop(
    columns=[col for col in constant_columns if col in clean_financials_df.columns]
)

print(f"Original dataset shape: {financials_df.shape}")
print(f"Cleaned dataset shape: {clean_financials_df.shape}")

clean_financials_df.head()

Original dataset shape: (4576, 360)
Cleaned dataset shape: (4576, 222)


,Symbol,Company Name,Exchange,Sector,Industry,Employees,MarketCap,EnterpriseValue,Income_Tax Effect Of Unusual Items,Income_Tax Rate For Calcs,Income_Normalized EBITDA,Income_Total Unusual Items,Income_Total Unusual Items Excluding Goodwill,Income_Net Income From Continuing Operation Net Minority Interest,Income_Reconciled Depreciation,Income_Reconciled Cost Of Revenue,Income_EBITDA,Income_EBIT,Income_Net Interest Income,Income_Interest Expense,Income_Interest Income,Income_Normalized Income,Income_Net Income From Continuing And Discontinued Operation,Income_Total Expenses,Income_Total Operating Income As Reported,Income_Diluted Average Shares,Income_Basic Average Shares,Income_Diluted EPS,Income_Basic EPS,Income_Diluted NI Availto Com Stockholders,Income_Net Income Common Stockholders,Income_Net Income,Income_Net Income Including Noncontrolling Interests,Income_Net Income Continuous Operations,Income_Tax Provision,Income_Pretax Income,Income_Other Income Expense,Income_Other Non Operating Income Expenses,Income_Special Income Charges,Income_Other Special Charges,Income_Impairment Of Capital Assets,Income_Gain On Sale Of Security,Income_Net Non Operating Interest Income Expense,Income_Interest Expense Non Operating,Income_Interest Income Non Operating,Income_Operating Income,Income_Operating Expense,Income_Depreciation Amortization Depletion Income Statement,Income_Depreciation And Amortization In Income Statement,Income_Research And Development,Income_Selling General And Administration,Income_Selling And Marketing Expense,Income_General And Administrative Expense,Income_Other Gand A,Income_Gross Profit,Income_Cost Of Revenue,Income_Total Revenue,Income_Operating Revenue,Balance_Treasury Shares Number,Balance_Ordinary Shares Number,Balance_Share Issued,Balance_Net Debt,Balance_Total Debt,Balance_Tangible Book Value,Balance_Invested Capital,Balance_Working Capital,Balance_Net Tangible Assets,Balance_Capital Lease Obligations,Balance_Common Stock Equity,Balance_Total Capitalization,Balance_Total Equity Gross Minority Interest,Balance_Stockholders Equity,Balance_Gains Losses Not Affecting Retained Earnings,Balance_Other Equity Adjustments,Balance_Retained Earnings,Balance_Additional Paid In Capital,Balance_Capital Stock,Balance_Common Stock,Balance_Preferred Stock,Balance_Total Liabilities Net Minority Interest,Balance_Total Non Current Liabilities Net Minority Interest,Balance_Other Non Current Liabilities,Balance_Non Current Deferred Liabilities,Balance_Non Current Deferred Taxes Liabilities,Balance_Long Term Debt And Capital Lease Obligation,Balance_Long Term Capital Lease Obligation,Balance_Long Term Debt,Balance_Current Liabilities,Balance_Other Current Liabilities,Balance_Current Deferred Liabilities,Balance_Current Deferred Revenue,Balance_Current Debt And Capital Lease Obligation,Balance_Current Debt,Balance_Other Current Borrowings,Balance_Payables And Accrued Expenses,Balance_Current Accrued Expenses,Balance_Payables,Balance_Total Tax Payable,Balance_Income Tax Payable,Balance_Accounts Payable,Balance_Total Assets,Balance_Total Non Current Assets,Balance_Other Non Current Assets,Balance_Non Current Deferred Assets,Balance_Non Current Deferred Taxes Assets,Balance_Goodwill And Other Intangible Assets,Balance_Other Intangible Assets,Balance_Goodwill,Balance_Net PPE,Balance_Accumulated Depreciation,Balance_Gross PPE,Balance_Leases,Balance_Other Properties,Balance_Machinery Furniture Equipment,Balance_Properties,Balance_Current Assets,Balance_Other Current Assets,Balance_Receivables,Balance_Taxes Receivable,Balance_Accounts Receivable,Balance_Allowance For Doubtful Accounts Receivable,Balance_Gross Accounts Receivable,Balance_Cash Cash Equivalents And Short Term Investments,Balance_Cash And Cash Equivalents,CashFlow_Free Cash Flow,CashFlow_Repurchase Of Capital Stock,CashFlow_Repayment Of Debt,CashFlow_Issuance Of Debt,CashFlow_Capital Expenditure,CashFlow_Interest Paid Supplemental Data,CashFlow_Income Tax Paid Supplementa

We lost 138 columns. This is substantial, but still within the expected range

### **3.5 Drop Prefix from Column Names**

In [23]:
clean_financials_df.columns = (
    clean_financials_df.columns
    .str.replace("Income_", "", regex=False)
    .str.replace("Balance_", "", regex=False)
    .str.replace("CashFlow_", "", regex=False)
)

clean_financials_df.head()

,Symbol,Company Name,Exchange,Sector,Industry,Employees,MarketCap,EnterpriseValue,Tax Effect Of Unusual Items,Tax Rate For Calcs,Normalized EBITDA,Total Unusual Items,Total Unusual Items Excluding Goodwill,Net Income From Continuing Operation Net Minority Interest,Reconciled Depreciation,Reconciled Cost Of Revenue,EBITDA,EBIT,Net Interest Income,Interest Expense,Interest Income,Normalized Income,Net Income From Continuing And Discontinued Operation,Total Expenses,Total Operating Income As Reported,Diluted Average Shares,Basic Average Shares,Diluted EPS,Basic EPS,Diluted NI Availto Com Stockholders,Net Income Common Stockholders,Net Income,Net Income Including Noncontrolling Interests,Net Income Continuous Operations,Tax Provision,Pretax Income,Other Income Expense,Other Non Operating Income Expenses,Special Income Charges,Other Special Charges,Impairment Of Capital Assets,Gain On Sale Of Security,Net Non Operating Interest Income Expense,Interest Expense Non Operating,Interest Income Non Operating,Operating Income,Operating Expense,Depreciation Amortization Depletion Income Statement,Depreciation And Amortization In Income Statement,Research And Development,Selling General And Administration,Selling And Marketing Expense,General And Administrative Expense,Other Gand A,Gross Profit,Cost Of Revenue,Total Revenue,Operating Revenue,Treasury Shares Number,Ordinary Shares Number,Share Issued,Net Debt,Total Debt,Tangible Book Value,Invested Capital,Working Capital,Net Tangible Assets,Capital Lease Obligations,Common Stock Equity,Total Capitalization,Total Equity Gross Minority Interest,Stockholders Equity,Gains Losses Not Affecting Retained Earnings,Other Equity Adjustments,Retained Earnings,Additional Paid In Capital,Capital Stock,Common Stock,Preferred Stock,Total Liabilities Net Minority Interest,Total Non Current Liabilities Net Minority Interest,Other Non Current Liabilities,Non Current Deferred Liabilities,Non Current Deferred Taxes Liabilities,Long Term Debt And Capital Lease Obligation,Long Term Capital Lease Obligation,Long Term Debt,Current Liabilities,Other Current Liabilities,Current Deferred Liabilities,Current Deferred Revenue,Current Debt And Capital Lease Obligation,Current Debt,Other Current Borrowings,Payables And Accrued Expenses,Current Accrued Expenses,Payables,Total Tax Payable,Income Tax Payable,Accounts Payable,Total Assets,Total Non Current Assets,Other Non Current Assets,Non Current Deferred Assets,Non Current Deferred Taxes Assets,Goodwill And Other Intangible Assets,Other Intangible Assets,Goodwill,Net PPE,Accumulated Depreciation,Gross PPE,Leases,Other Properties,Machinery Furniture Equipment,Properties,Current Assets,Other Current Assets,Receivables,Taxes Receivable,Accounts Receivable,Allowance For Doubtful Accounts Receivable,Gross Accounts Receivable,Cash Cash Equivalents And Short Term Investments,Cash And Cash Equivalents,Free Cash Flow,Repurchase Of Capital Stock,Repayment Of Debt,Issuance Of Debt,Capital Expenditure,Interest Paid Supplemental Data,Income Tax Paid Supplemental Data,End Cash Position,Beginning Cash Position,Effect Of Exchange Rate Changes,Changes In Cash,Financing Cash Flow,Cash Flow From Continuing Financing Activities,Net Other Financing Charges,Proceeds From Stock Option Exercised,Net Common Stock Issuance,Common Stock Payments,Net Issuance Payments Of Debt,Net Short Term Debt Issuance,Net Long Term Debt Issuance,Long Term Debt Payments,Long Term Debt Issuance,Investing Cash Flow,Cash Flow From Continuing Investing Activities,Net Business Purchase And Sale,Sale Of Business,Purchase Of Business,Net PPE Purchase And Sale,Purchase Of PPE,Capital Expenditure Reported,Operating Cash Flow,Cash Flow From Continuing Operating Activities,Change In Working Capital,Change In Other Working Capital,Change In Other Current Assets,Change In Payables And Accrued Expense,Change In Accrued Expense,Change In Payable,Change In Account Payable,Change In Tax Payable,Change In Income Tax Payable,Change

# **4. Final Check**

In [24]:
print(f"Final dataset shape: {clean_financials_df.shape}")
print(f"Remaining missing values: {clean_financials_df.isna().sum().sum():,}")

clean_financials_df.isna().mean().sort_values(ascending=False).head(20)

Final dataset shape: (4576, 222)
Remaining missing values: 304,127


Change In Income Tax Payable               0.796547
Change In Tax Payable                      0.796547
Non Current Deferred Revenue               0.786276
Interest Payable                           0.786058
Earnings Losses From Equity Investments    0.776442
Purchase Of Intangibles                    0.774257
Long Term Equity Investment                0.763986
Net Intangibles Purchase And Sale          0.757430
Other Special Charges                      0.757212
Capital Expenditure Reported               0.749344
Sale Of Business                           0.743663
Salaries And Wages                         0.741259
Work In Process                            0.740822
Cash Financial                             0.739073
Other Payable                              0.734266
Taxes Receivable                           0.733173
Restricted Cash                            0.719187
Impairment Of Capital Assets               0.717657
Other Operating Expenses                   0.698427
Net Short Te

In [25]:
output_path = INTERIM_DATA_DIR / "clean_financials_with_metadata.xlsx"

clean_financials_df.to_excel(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: C:\Users\semoy\OneDrive\Documentos\GitHub\financial_kpi_clustering\data\interim\clean_financials_with_metadata.xlsx


# Notebook Summary

This notebook cleaned and prepared the structured financial dataset generated in Notebook 1 for downstream KPI engineering and clustering analysis.

The process began by loading the financial statement dataset enriched with company metadata, including sector, industry, employee count, market capitalisation, enterprise value, and currency. Initial validation was performed to inspect dataset size, column structure, missing values, and financial feature coverage.

Sparse financial features were removed using a missing-value threshold, ensuring that the retained variables have sufficient coverage across the company universe. Constant or non-informative columns were also identified and removed to reduce noise in later analysis. Metadata fields were retained where they provide useful context for sector-level segmentation and cluster interpretation.

Finally, financial statement prefixes were removed from column names to improve readability, and the cleaned dataset was exported for use in Notebook 3. The resulting dataset provides a more reliable foundation for KPI engineering, feature scaling, dimensionality reduction, and clustering experiments.